# ROBUST04 Phase 1: Pure RRF Optimized (Before SPLADE)

## Based on: Copy of ROBUST04_Run3_Ensemble - Copy (1).ipynb

## Fixes Applied:
1. ✅ FIX #1: RM3 parameters improved (fb_docs 10→20, fb_terms 10→15, weight 0.5→0.6)
2. ✅ FIX #2: k_rrf default changed from 60→10 (optimal from tuning)
3. ✅ FIX #3: Test rerank_depth changed from 100→1000 (consistency!)
4. ✅ FIX #4: Removed unused functions
5. ✅ FIX #5: Fixed misleading SPLADE comments

## Validated Performance:
- Baseline (k=60): ~0.27
- Optimized (k=10): 0.2769
- Target with fixes: 0.278-0.280

In [ ]:
hugging_face_token = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login
login(hugging_face_token)

In [ ]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Checking Java version...")

try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 ready!")
print("="*70)

In [ ]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")

!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy
!pip install faiss-cpu --no-cache

print("✓ All packages installed!")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pyserini.encode import SpladeQueryEncoder
import warnings
import time

warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive, userdata
import os
import sys

try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                os.chdir(path)
                BASE_DIR = path
                break
else:
    BASE_DIR = os.getcwd()

print(f"\n📁 Checking for required files...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    print(f"  ✓ Found: queriesROBUST.txt")
if os.path.exists('qrels_50_Queries'):
    print(f"  ✓ Found: qrels_50_Queries")

print("\n" + "="*70)
print("✓ Setup complete!")
print("="*70)

## 1. Configuration

In [ ]:
# Model configuration
USE_MONOT5 = True

print(f"\n🎯 Phase 1 Configuration:")
print(f"  • MonoT5 Reranker: {USE_MONOT5}")
print(f"  • Pure RRF (all weights = 1.0)")
print(f"  • k_rrf: 10 (FIX #2)")
print(f"  • rerank_depth: 1000")

## 2. Load Index and Configure Searchers

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

# FIX #1: RM3 searcher with IMPROVED parameters
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
# OLD: fb_docs=10, fb_terms=10, original_query_weight=0.5
# NEW: fb_docs=20, fb_terms=15, original_query_weight=0.6
rm3_searcher.set_rm3(fb_docs=20, fb_terms=15, original_query_weight=0.6)

searchers = [bm25_searcher, rm3_searcher]

print("✓ Configured BM25 + RM3")
print("  BM25: k1=0.9, b=0.4")
print("  RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6 (FIX #1)")

## 3. Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## 4. Helper Functions

In [ ]:
# FIX #4: Removed unused functions (reciprocal_rank_fusion, get_adaptive_weights, ensemble_fusion)

def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## 5. Multi-Method Retrieval

In [ ]:
def multi_method_fusion(query_text, k=60):
    """
    Get results from BM25 and RM3, return both separately for RRF.
    """
    bm25_hits = searchers[0].search(query_text, k=1000)
    rm3_hits = searchers[1].search(query_text, k=1000)

    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    return bm25_results, rm3_results

## 6. Load MonoT5 Reranker

In [ ]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  ✓ MonoT5 loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## 7. MonoT5 Reranker Function

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=8):
    """MonoT5 reranking (pointwise scoring)

    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        print("  Warning: No documents retrieved for reranking")
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()
                all_scores.extend(batch_scores)
        except Exception as e:
            print(f"  Batch error: {e}")
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## 10. Phase 1 Pipeline (Pure RRF)

In [ ]:
# FIX #2: Changed default k_rrf from 60 to 10
def ultimate_pipeline(query, rerank_depth=1000, k_rrf=10):
    """
    Phase 1: Pure RRF with Optimized Parameters

    BM25 + RM3 + MonoT5 (all weights = 1.0)
    FIX #2: k_rrf default changed from 60 to 10
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_fusion(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        top_k_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_k_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: Pure RRF Fusion (all weights = 1.0)
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        # BM25 contribution (weight = 1.0)
        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        # RM3 contribution (weight = 1.0)
        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        # MonoT5 contribution (weight = 1.0)
        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score (descending)
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results

## Evaluate on Training Set

In [ ]:
print("="*70)
print("📊 EVALUATING PHASE 1 ON TRAINING SET")
print("="*70)
print("Purpose: Validate fixes and improvements")
print("="*70)
print()

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = ultimate_pipeline(query_text, rerank_depth=1000)

        for docid, score, rank in results:
            if isinstance(score, np.ndarray):
                score = float(score.item())
            elif isinstance(score, list):
                score = float(score[0]) if score else 0.0
            else:
                score = float(score)

            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  ✓ Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("📈 COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("🏆 PHASE 1 RESULTS - TRAINING SET (50 queries)")
print("="*70)
print("\n📊 Your Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  ← Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Compare
baseline_map = 0.2769
current_map = results_metrics[MAP]

print(f"\n📈 Comparison:")
print(f"  Previous (k=10, old RM3): {baseline_map:.4f}")
print(f"  Current (k=10, new RM3):  {current_map:.4f}")

if current_map >= baseline_map:
    improvement = current_map - baseline_map
    print(f"  ✅ Improvement: +{improvement:.4f} ({improvement/baseline_map*100:+.2f}%)")
else:
    decline = baseline_map - current_map
    print(f"  ⚠️  Decline: -{decline:.4f}")

print(f"\n⏱️  Time: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

print("\n" + "="*70)
if current_map >= 0.278:
    print("✅ EXCELLENT! Phase 1 optimized successfully!")
elif current_map >= 0.275:
    print("✓ GOOD! Phase 1 is solid")
else:
    print("📝 Results logged. Ready for Phase 2.")
print("="*70)

## Generate Test Results

In [ ]:
# Check which models are active
monot5_active = 'monot5_model' in globals() and monot5_model is not None

print("="*70)
print("🚀 GENERATING TEST RESULTS - PHASE 1")
print("="*70)
print(f"Configuration:")
print(f"  • MonoT5 reranking: {monot5_active}")
# FIX #5: Removed misleading SPLADE mention
print(f"  • Rerank depth: 1000 documents (FIX #3)")
print(f"  • k_rrf: 10 (FIX #2)")
print(f"  • Total test queries: {len(test_qids)}")
print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()

    # FIX #3: Use rerank_depth=1000 instead of 100
    results = ultimate_pipeline(query_text, rerank_depth=1000)

    query_time = time.time() - query_start

    for docid, score, rank in results:
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)

        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("─"*70)
        print(f"📊 PROGRESS: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f} min, Remaining: ~{remaining/60:.1f} min")
        print(f"  Avg: {avg_time:.1f}s/query")
        print("─"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(run3_results):,}")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Avg: {total_time/len(test_qids):.1f}s/query")
print("="*70)

## 11. Save Results

In [ ]:
with open('run_3_phase1_fixed.res', 'w') as f:
    for r in run3_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_phase1\n")

print("✓ Saved to run_3_phase1_fixed.res")
print("\nFirst 5 lines:")
with open('run_3_phase1_fixed.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("📝 PHASE 1 COMPLETE!")
print("="*70)
print("\n🔧 All Fixes Applied:")
print("  ✅ FIX #1: RM3 parameters improved")
print("  ✅ FIX #2: k_rrf default = 10 (was 60)")
print("  ✅ FIX #3: Test rerank_depth = 1000 (was 100)")
print("  ✅ FIX #4: Removed unused functions")
print("  ✅ FIX #5: Fixed SPLADE comments")
print("\n📈 Performance:")
print(f"  Training MAP: {results_metrics[MAP]:.4f}")
print(f"  Baseline: 0.2769")
print("\n🚀 Ready for Phase 2: Add SPLADE")
print("="*70)